# Text Preprocessing & Normalization: clean, normalize, then measure

Open any real text dataset and the first thing that hits you is the *mess*: the same word arrives as `Apple`, `apple`, `APPLE`, and `apple,`; HTML tags and URLs nobody wants; smart quotes, emoji, accented characters typed three different ways. A model that has never seen this raw soup sees **four different tokens where a human sees one word** — and every spurious distinction splits the signal.

**Preprocessing** is the disciplined cleanup that fixes this. Its whole job fits in one sentence worth memorizing: **reduce sparsity while preserving signal.** This notebook builds the classical pipeline step by step, *measures* what each step buys, and shows the one place where the textbook pipeline is exactly the wrong move (a transformer).

Every function here is imported from `text_preprocessing.py` — the same code that generates the page's figures — so what you run is what you read. This chapter is **pure Python/numpy** (no tensors, no GPU): it runs anywhere in a few seconds.

> **Step 0 — environment + the data.** This is a CPU-only chapter, so we print an honest device line (`device: cpu (pure-Python/numpy)`). We load NLTK's stopword list + stemmer + lemmatizer and the 4-category 20 Newsgroups corpus once, here, and reuse them throughout.

In [1]:
import nltk
from text_preprocessing import device_line, load_newsgroups

for pkg in ('stopwords', 'wordnet', 'omw-1.4'):
    nltk.download(pkg, quiet=True)
from nltk.corpus import stopwords

STOP = set(stopwords.words('english'))
train_docs, train_labels, test_docs, test_labels = load_newsgroups()

assert len(STOP) > 100, 'expected a real English stopword list'
assert len(train_docs) > 2000, 'expected the 4-category 20 Newsgroups train split'
print(device_line())
print(f'stopwords: {len(STOP)} words   |   train docs: {len(train_docs)}')

device: cpu (pure-Python/numpy)   numpy: 2.4.6   torch: 2.12.0 (imported for detection only, unused)
stopwords: 198 words   |   train docs: 2321


## 1. The problem, made concrete

Here is one ordinary line of user-generated text. Count the distinct kinds of *noise* the model would otherwise have to learn around: markup, shouting case, `!!!`, an em-dash, a URL, an emoji, a smart apostrophe, an accented `é`, a clipped contraction (`lovin'`), a hashtag.

In [2]:
messy = "<p>HELLO!!!</p> Visit https://x.io \u2014 I\u2019m lovin\u2019 caf\u00e9s & NLP \U0001F600 #AI"
print(repr(messy))
print('raw character length:', len(messy))

'<p>HELLO!!!</p> Visit https://x.io — I’m lovin’ cafés & NLP 😀 #AI'
raw character length: 65


> **Step 1 — run the full classical pipeline.** `preprocess` applies the canonical ordered pipeline: strip HTML → unescape entities → URL/emoji → placeholders → NFKC + lowercase → fold accents → drop punctuation → collapse whitespace → tokenize → drop stopwords. **Order is load-bearing** (strip HTML *before* lowercasing; normalize Unicode *before* tokenizing). We assert the URL became a reusable `<url>` placeholder (not deleted) and that the pronoun `i` was dropped as a stopword, then print the canonical tokens.

In [3]:
from text_preprocessing import preprocess

tokens = preprocess(messy, stopwords_set=STOP)
assert '<url>' in tokens and 'url' not in tokens, 'URL should be a placeholder, not stripped'
assert 'i' not in tokens, "the pronoun 'i' is a stopword and should be removed"
print('clean tokens ->', tokens)

clean tokens -> ['hello', 'visit', '<url>', 'lovin', 'cafes', 'nlp', '<emoji>', 'ai']


Watch what each stage *bought*: layout removed, a unique URL and an emoji collapsed into reusable placeholders, casing and Unicode merged, `cafés`→`cafes` (so it matches `cafe` elsewhere), a contentless pronoun dropped. A high-variance string became a short list of canonical tokens — **reduced sparsity**. But note we *also* lost the `!!!` emphasis and the emoji's sentiment direction — **a real signal cost** we'd avoid on a sentiment task.

## 2. Why preprocessing exists at all: Zipf's law

Preprocessing is motivated by a startlingly regular fact about language. **Zipf's law:** rank every word by frequency; the $r$-th most frequent word has frequency roughly

$$f(r) \;\approx\; \frac{C}{r^{s}}, \qquad s \approx 1.$$

Take logs and it's a straight line: $\log f(r) = \log C - s\,\log r$. So a tiny number of words are wildly frequent (the slope is steep at the head) and a **huge** tail of words each occur once or twice. Two consequences fall straight out, and they *are* the reason preprocessing works:
1. the **head** is dominated by low-content function words (`the`, `of`, `a`) → *stopword removal*;
2. the **tail** is full of rare inflected/typo'd variants → *stemming, lemmatization, normalization* collapse them.

> **Step 2 — fit the Zipf slope on the real corpus.** We tokenize 20 Newsgroups, rank the words, and fit the log-log line. We assert the measured slope sits near $-1$ (the Zipf signature), then print it and the top words.

In [4]:
from text_preprocessing import zipf_rank_frequency

ranks, freqs, slope, intercept, top = zipf_rank_frequency(train_docs)
assert -1.2 < slope < -0.7, f'a natural corpus should show a Zipf slope near -1, got {slope:.3f}'
print(f'measured Zipf slope (ranks 1..1000): {slope:.3f}   (ideal -1)')
print('top 10 words:', [w for w, _ in top])
total = freqs.sum()
share_10 = freqs[:10].sum() / total * 100
share_100 = freqs[:100].sum() / total * 100
print(f'top 10 words   = {share_10:.0f}% of all tokens')
print(f'top 100 words  = {share_100:.0f}% of all tokens')

measured Zipf slope (ranks 1..1000): -0.948   (ideal -1)
top 10 words: ['the', 'to', 'of', 'a', 'and', 'i', 'in', 'is', 'that', 'it']
top 10 words   = 19% of all tokens
top 100 words  = 44% of all tokens


The top 10 words — every one a stopword — account for ~19% of *all* tokens; the top 100 account for ~44%. That is why stopword removal is so cheap and so effective: a list of a few hundred words strips out a huge fraction of the token stream while deleting almost none of the *meaning* (for topic tasks). The figure `tp_zipf.png` plots this exact line.

## 3. The sparsity lever, measured: vocabulary shrinks at every step

> **Step 3 — count the distinct vocabulary after each cleaning step.** `vocab_shrink_stages` runs the pipeline on the whole corpus and reports the number of *distinct* tokens after each step. We assert the sequence is monotonically non-increasing (no step may *grow* the vocab), then print it. This is the quantitative meaning of 'reduce sparsity'.

In [5]:
from text_preprocessing import vocab_shrink_stages

stages = vocab_shrink_stages(train_docs, stopwords_set=STOP)
sizes = [size for _, size in stages]
assert sizes == sorted(sizes, reverse=True), 'each step must not grow the vocabulary'
for name, size in stages:
    print(f'{name.replace(chr(10), " "):28s} {size:6,d}')
print(f'\noverall: {sizes[0]:,} -> {sizes[-1]:,}  ({sizes[0] / sizes[-1]:.1f}x smaller)')

raw (case-sensitive)         61,296
lowercase                    55,173
+ tokenize + NFKC            30,733
+ stopword removal           30,582
+ Porter stemming            23,365

overall: 61,296 -> 23,365  (2.6x smaller)


The biggest single cut is **tokenization + NFKC** (it removes the case-and-punctuation explosion of raw whitespace splitting), and **stemming** delivers the second big cut by merging inflected forms. The corresponding figure is `tp_vocab_shrink.png`.

## 4. Stemming vs lemmatization — the headline comparison

Both reduce inflected forms to a common base so `study`, `studies`, `studying` count as one. They do it very differently:

- **Stemming** (Porter, 1980) is **crude rule-based suffix stripping** — fast, no dictionary, but blind to meaning, so it produces non-words (`studies`→`studi`) and over-stems (`organization`→`organ`).
- **Lemmatization** is **dictionary + POS-aware** — it returns the real lemma (`better`→`good`, `are`→`be`, `mice`→`mouse`), at the cost of speed and needing the POS.

> **Step 4 — run both on tricky words.** `stem_vs_lemma` returns the *real* NLTK outputs. We assert the two methods **disagree** on the irregular/tricky cases (the whole point), then print the comparison with the disagreements flagged.

In [6]:
from text_preprocessing import stem_vs_lemma

pairs = [('studies','n'), ('studying','v'), ('better','a'), ('best','a'),
         ('are','v'), ('is','v'), ('mice','n'), ('organization','n'), ('running','v')]
rows = stem_vs_lemma(pairs)
disagreements = sum(1 for *_, agree in rows if not agree)
assert disagreements >= 5, 'stemmer and lemmatizer should disagree on most irregulars'
print(f"{'word':13s}{'POS':5s}{'Porter stem':14s}{'WordNet lemma':15s}flag")
print('-' * 52)
for word, pos, stem, lemma, agree in rows:
    print(f'{word:13s}{pos:5s}{stem:14s}{lemma:15s}{"agree" if agree else "DIFFER"}')

word         POS  Porter stem   WordNet lemma  flag
----------------------------------------------------
studies      n    studi         study          DIFFER
studying     v    studi         study          DIFFER
better       a    better        good           DIFFER
best         a    best          best           agree
are          v    are           be             DIFFER
is           v    is            be             DIFFER
mice         n    mice          mouse          DIFFER
organization n    organ         organization   DIFFER
running      v    run           run            agree


The pattern is unmistakable: they **agree** on simple regular suffixes (`running`→`run`) and **disagree** on (a) irregulars (`better`/`good`, `mice`/`mouse`, `are`/`be`), (b) over-stems (`organization`/`organ`), and (c) outright garbage stems (`studi`). Rule of thumb: **stem** when speed/recall rule and ugliness is fine (large-scale search); **lemmatize** when you need real words and correctness. Figure: `tp_stem_vs_lemma.png`.

## 5. The stopword trap: removing `not` can flip a label

Stopword removal is the right move for topic tasks — but function words are **signal** for many others, and the classic landmine is **negation**.

> **Step 5 — watch de-stopwording destroy negation.** `not` is on the standard stoplist. We assert it gets removed from *'this movie is not good'*, leaving a **positive** bag-of-words for a **negative** review — the cleaning *inverted the sentiment*.

In [7]:
sentence = 'this movie is not good'
naive = [t for t in sentence.split() if t not in STOP]
assert 'not' not in naive, "'not' is a stopword; removing it inverts the sentiment"
print('original :', sentence)
print('de-stopworded ->', naive, '  <- the negation is gone; this now looks POSITIVE')

original : this movie is not good
de-stopworded -> ['movie', 'good']   <- the negation is gone; this now looks POSITIVE


That single line is the whole argument against blindly de-stopwording a sentiment task. The same operation that *helped* topic classification (Step 3) would *hurt* here. **Every preprocessing step is a bet that the variation it collapses is noise** — and that bet flips with the task.

## 6. Unicode normalization: identical-looking text, different bytes

> **Step 6 — the same-looking string is two different strings until you normalize.** `café` can be `cafe` + a combining accent (5 code points) or a precomposed `é` (4 code points). We assert they are **unequal raw** but **equal after NFC**, then show NFKC additionally folding compatibility look-alikes (the `ﬁ` ligature, the `½` fraction).

In [8]:
from text_preprocessing import unicode_demo

uni = unicode_demo()
assert uni['equal_raw'] is False, 'the two spellings are different byte sequences'
assert uni['equal_after_nfc'] is True, 'NFC must merge the canonical duplicates'
print('café (combining) vs café (precomposed):')
print('  equal raw? ', uni['equal_raw'], ' | equal after NFC? ', uni['equal_after_nfc'])
print('  code points:', uni['combining_codepoints'], 'vs', uni['precomposed_codepoints'])
print('NFKC folds compatibility look-alikes:')
print('  NFKC(ﬁle) ->', repr(uni['nfkc_ligature_file']))
print('  NFKC(½)   ->', repr(uni['nfkc_half']))
print('  NFKC(Ⅸ)   ->', repr(uni['nfkc_roman_nine']))

café (combining) vs café (precomposed):
  equal raw?  False  | equal after NFC?  True
  code points: 5 vs 4
NFKC folds compatibility look-alikes:
  NFKC(ﬁle) -> 'file'
  NFKC(½)   -> '1⁄2'
  NFKC(Ⅸ)   -> 'IX'


Without normalization, `café` typed two ways are two vocabulary entries with split counts and silently-missed search hits. **NFC** is the safe default (merges true duplicates, stays human-faithful); **NFKC** also folds formatting variants (the right default for search, but lossy: `²`→`2`). Figure: `tp_unicode_norm.png`.

## 7. The payoff, measured: heavy cleaning helps a *classical* model

Numbers beat assertions. On 20 Newsgroups we train TF-IDF + logistic regression under three increasingly aggressive cleaning configs and measure **vocabulary** and **test accuracy**.

> **Step 7 — the measured sweep.** `classifier_sweep` runs raw → +stopwords → +stemming. We assert stemming **shrinks** the TF-IDF vocabulary, then print vocab and accuracy for each. (LogReg is seeded, so this is reproducible.)

In [9]:
from text_preprocessing import classifier_sweep

sweep = classifier_sweep(train_docs, train_labels, test_docs, test_labels, stopwords_set=STOP)
raw_vocab, stem_vocab = sweep[0][1], sweep[-1][1]
assert stem_vocab < raw_vocab, 'stemming must shrink the TF-IDF vocabulary'
print(f"{'config':22s}{'vocab':>8s}{'accuracy':>11s}")
print('-' * 41)
for name, vocab, acc in sweep:
    print(f'{name:22s}{vocab:>8,d}{acc * 100:>10.1f}%')
print(f'\nstemming cut the vocab {raw_vocab / stem_vocab:.1f}x at equal-or-better accuracy')

config                   vocab   accuracy
-----------------------------------------
raw (lowercase)         12,399      87.6%
+ stopword removal      12,259      88.3%
+ Porter stemming        8,814      88.5%

stemming cut the vocab 1.4x at equal-or-better accuracy


Read it carefully: **stopword removal** barely changes the vocab *count* (stopwords are few distinct words, just very frequent) but nudges accuracy **up** — it removed high-magnitude, low-information dimensions. **Stemming** is the big vocabulary cut (~1.4×) by merging inflected forms, and accuracy holds or ticks up. On a **topic** task with a **classical** model, aggressive cleaning is a measurable win — *this is the case where the textbook pipeline earns its keep*. Figure: `tp_classical_vs_neural.png`.

> **The crucial caveat:** re-run this on a **sentiment** task and remove `not` (Step 5) and you'd watch accuracy *drop*. And every one of these steps would *hurt* a transformer, which wants the case, punctuation, and morphology back. **Match the preprocessing to the consumer.**

## What we saw

- **Zipf's law** (slope ≈ −0.95, measured) is *why* preprocessing works: a few function words dominate the head (→ stopword removal) and a long tail of rare variants fills the bottom (→ stemming/normalization).
- **Every cleaning step shrank the vocabulary** monotonically — the quantitative meaning of *reduce sparsity* (≈ 61k → 23k distinct tokens).
- **Stemming is fast and crude** (`studi`, `organ`); **lemmatization is slower and correct** (`good`, `be`, `mouse`) — they disagree on exactly the irregular/over-stem cases.
- **De-stopwording destroyed a negation** (`not good` → `good`), and **Unicode normalization** quietly merged byte-level duplicates you'd otherwise never notice.
- **Measured:** heavy cleaning shrank a TF-IDF vocab ~1.4× at equal-or-better accuracy on a topic task — a real win for a classical model, and the wrong move for a transformer.

**The one-sentence takeaway:** preprocessing exists to **reduce sparsity while preserving signal** — and the right *amount* depends entirely on the consumer (heavy for bag-of-words, almost none for a pretrained transformer).